# Make a wrapped heatmap with multiple rows
This notebook makes wrapped heatmaps with multiple rows, designed for use as paper figures.

In [ ]:
import altair as alt

import pandas as pd

_ = alt.data_transformers.disable_max_rows()

## Get configuration parameters

We get the parameters passed by `snakemake`:

In [ ]:
# Parameters - this cell will be replaced by papermill
data_csv = None
chart_html = None
chart_svg = None
chart_png = None
title = None
effect_col = None
data_query_str = ""
sites_per_row = 126
site_label_freq = 20
site_label_start = None
color_scheme = "redblue"
fixed_min = -4
fixed_max = 2
alphabet = "RKHDEQNSTYWFAILMVGPC"
dark_gray_muts = None

## Read the input data

In [ ]:
if isinstance(alphabet, str):
    alphabet = list(alphabet)

data = pd.read_csv(data_csv, dtype={"site": str})
print(f"Read {len(data)=} with {data.columns=}")

# Add sequential_site column if it doesn't exist
if "sequential_site" not in data.columns:
    # Create sequential numbering based on site order
    # Sort sites treating them as strings with natural sort (e.g., "1", "2", "10", "158a")
    import re
    def natural_sort_key(s):
        """Sort key that handles alphanumeric strings like '158a'"""
        return [int(text) if text.isdigit() else text.lower() 
                for text in re.split('([0-9]+)', str(s))]
    
    unique_sites = sorted(data["site"].unique(), key=natural_sort_key)
    # Map each site to a sequential integer
    site_to_sequential = {site: i for i, site in enumerate(unique_sites)}
    data["sequential_site"] = data["site"].map(site_to_sequential)
    print(f"Added sequential_site column: {len(unique_sites)} unique sites")
    print(f"Site range: {unique_sites[0]} to {unique_sites[-1]}")

if data_query_str:
    data = data.query(data_query_str).reset_index(drop=True)
    print(f"After querying with {data_query_str=}, {len(data)=}")
    
req_cols = ["site", "sequential_site", "wildtype", "mutant", effect_col]
assert set(req_cols).issubset(data.columns), f"{data.columns=} lacks {req_cols=}"

if dark_gray_muts and (dark_gray_muts["col"] not in data.columns):
    raise ValueError(f"{dark_gray_muts['col']=} not in {data.columns=}")
elif dark_gray_muts and (dark_gray_muts["col"] not in req_cols):
    req_cols.append(dark_gray_muts["col"])

data = data[data["mutant"].isin(alphabet) & data["wildtype"].isin(alphabet)].reset_index(drop=True)
print(f"After getting just amino acids of {alphabet=}, {len(data)=}")

data = data[req_cols]
assert len(data) == len(data.groupby(["site", "wildtype", "mutant"]))

## Make heatmap

In [ ]:
heatmap_base = (
    alt.Chart(data)
    .encode(alt.Y("mutant", sort=alphabet, title="amino acid"))
    .properties(width=alt.Step(9), height=alt.Step(9))
)

heatmap_bg = heatmap_base.transform_impute(
    impute="_stat_dummy",
    key="mutant",
    keyvals=alphabet,
    groupby=["site"],
    value=None,
).mark_rect(color="#E0E0E0", opacity=0.8)

heatmap_wildtype = (
    heatmap_base
    .transform_filter(alt.datum["wildtype"] == alt.datum["mutant"])
    .mark_text(text="x", color="black")
)

heatmap_muts = (
    heatmap_base
    .encode(
        alt.Color(
            effect_col,
            scale=alt.Scale(
                scheme=color_scheme,
                domainMid=0,
                domainMin=fixed_min,
                domainMax=fixed_max,
                clamp=True,
            ),
        ),
        tooltip=["site", "mutant", "wildtype", alt.Tooltip(effect_col, format=".2f")],
    )
    .mark_rect(stroke="black", opacity=1, strokeOpacity=1)
)

if dark_gray_muts:
    heatmap_muts = heatmap_muts.transform_filter(
        alt.datum[dark_gray_muts["col"]] >= dark_gray_muts["cutoff"]
    )

    heatmap_dark_gray = (
        heatmap_base
        .transform_filter(alt.datum[dark_gray_muts["col"]] < dark_gray_muts["cutoff"])
        .transform_calculate(filtered="0")
        .mark_rect(stroke="black", opacity=1, strokeOpacity=1, color="silver")
    )

heatmap_rows = []
sequential_sites = sorted(data["sequential_site"].unique())
for i in range(0, len(sequential_sites), sites_per_row):
    row_sites = sequential_sites[i: i + sites_per_row]
    last_row = row_sites[-1] == sequential_sites[-1]
    sequential_to_site = data.set_index("sequential_site")["site"].to_dict()
    
    # Get all actual site labels for this row
    row_site_labels = [sequential_to_site[seq] for seq in row_sites]
    
    # label only some of the sites, starting at site_label_start and every site_label_freq
    if site_label_start is not None:
        # Calculate which sequential sites to label
        to_label_sequential = []
        current_label = site_label_start
        while current_label <= row_sites[-1]:
            if current_label >= row_sites[0]:  # only if in current row
                to_label_sequential.append(current_label)
            current_label += site_label_freq
        to_label_values = [
            sequential_to_site[seq_site]
            for seq_site in to_label_sequential
            if seq_site in sequential_to_site and sequential_to_site[seq_site] in row_site_labels
        ]
    else:
        # Default behavior: label from first site in row
        to_label_values = [
            sequential_to_site[seq]
            for seq in range(row_sites[0], row_sites[-1] + 1, site_label_freq)
            if seq in sequential_to_site and sequential_to_site[seq] in row_site_labels
        ]
    
    # Ensure we always have at least some labels
    if not to_label_values:
        # Use every Nth site from this row
        step = max(1, len(row_site_labels) // 5)  # show ~5 labels
        to_label_values = [row_site_labels[j] for j in range(0, len(row_site_labels), step)]
    
    if dark_gray_muts:
        row_charts = heatmap_bg + heatmap_dark_gray + heatmap_muts + heatmap_wildtype
    else:
        row_charts = heatmap_bg + heatmap_muts + heatmap_wildtype
    
    # Only specify values if we have them, otherwise let Altair auto-generate
    axis_kwargs = {"labelAngle": 0}
    if to_label_values:
        axis_kwargs["values"] = to_label_values
    
    heatmap_rows.append(
        row_charts
        .encode(
            alt.X(
                "site:N",
                title="site" if last_row else None,
                sort=alt.SortField("sequential_site"),
                scale=alt.Scale(nice=False, zero=False),
                axis=alt.Axis(**axis_kwargs)
            ),
        )
        .transform_filter(
            (alt.datum["sequential_site"] >= min(row_sites))
            & (alt.datum["sequential_site"] <= max(row_sites))
        )
    )

heatmap = (
    alt.vconcat(*heatmap_rows, spacing=6)
    .configure_axis(tickColor="black", tickSize=4, titleFontSize=16)
    .configure_legend(
        orient="bottom",
        gradientStrokeWidth=1,
        gradientStrokeColor="black",
        titleAnchor="middle",
        titleFontSize=16,
        titleLimit=200,
    )
    .properties(title=title)
    .configure_title(anchor="middle", fontSize=18)
)

print(f"Saving {chart_html=}")
heatmap.save(chart_html)

# Try to save SVG if requested - requires vl-convert-python
if chart_svg:
    try:
        print(f"Saving {chart_svg=}")
        heatmap.save(chart_svg)
    except ValueError as e:
        print(f"Warning: Could not save SVG: {e}")

# For PNG, convert from HTML using matplotlib if vl-convert-python is not available
if chart_png:
    try:
        print(f"Saving {chart_png=}")
        heatmap.save(chart_png)
    except (ValueError, ImportError) as e:
        print(f"vl-convert-python not available, trying alternative PNG conversion: {e}")
        # Alternative: use altair_saver or similar, or just skip
        print("Skipping PNG save - install vl-convert-python to enable PNG export")

heatmap